<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo">
    </a>
</p>

In [1]:

# Import required libraries

import pandas as pd
import dash
from dash import dcc
from dash import html
from dash.dependencies import Input, Output
import plotly.express as px


In [2]:

path = ''
spacex_data = pd.read_csv(path + 'spacex_launch_dash.csv')
spacex_df = pd.DataFrame(spacex_data)

# spacex_df.head()
spacex_df = spacex_df.iloc[:,1:7]
spacex_df.head()


,Flight Number,Launch Site,class,Payload Mass (kg),Booster Version,Booster Version Category
0,1,CCAFS LC-40,0,0.0,F9 v1.0 B0003,v1.0
1,2,CCAFS LC-40,0,0.0,F9 v1.0 B0004,v1.0
2,3,CCAFS LC-40,0,525.0,F9 v1.0 B0005,v1.0
3,4,CCAFS LC-40,0,500.0,F9 v1.0 B0006,v1.0
4,5,CCAFS LC-40,0,677.0,F9 v1.0 B0007,v1.0


In [3]:

spacex_df['Launch Site'].unique()


array(['CCAFS LC-40', 'VAFB SLC-4E', 'KSC LC-39A', 'CCAFS SLC-40'],
      dtype=object)

In [4]:

Min_Pld = int(spacex_df['Payload Mass (kg)'].min())
Max_Pld = int(spacex_df['Payload Mass (kg)'].max())
Pld_Step = int(Max_Pld/10)
pl = list(range(Min_Pld, Max_Pld + Pld_Step, Pld_Step))
lbls = list(range(0,12))
# print(pl, lbls)


In [5]:

# Create a dash application
app = dash.Dash(__name__)


TASK 3: Add a Range Slider to Select Payload.

In [6]:

# Task 3 layout
app.layout = html.Div(children=[html.H1('SpaceX Falcon_9 Success Launch Analytics', 
                                         style={'textAlign': 'center',
                                                'color':'#503D36',
                                                'font-size': 30}),
                               html.Div(["site-dropdown: ", 
                                         dcc.Dropdown(id='site-dropdown',
                                                      options=[{'label': 'All', 'value': 'ALL'},
                                                               {'label': 'CCAFS LC-40', 'value': 'CCAFS LC-40'},
                                                               {'label': 'CCAFS SLC-40', 'value': 'CCAFS SLC-40'},
                                                               {'label': 'KSC LC-39A', 'value': 'KSC LC-39A'},
                                                               {'label': 'VAFB SLC-4E', 'value': 'VAFB SLC-4E'},],
                                                      value='ALL', 
                                                      # placeholder="Choose a Launch Site", 
                                                      searchable=True),
                                         html.Div([html.Div(dcc.Graph(id='success-site_pie-chart'))],
                                                  style={'display':'flex'}),]),
                               html.Div(["payload-slider: ", 
                                         dcc.RangeSlider(id='payload-slider', 
                                                         min=Min_Pld, 
                                                         max=Max_Pld,
                                                         value=[Min_Pld, Max_Pld]
                                                        ), 
                                                         #updatemode='drag'), 
                                         html.Div([html.Div(dcc.Graph(id='success-payload_scatter-chart'))],
                                                  style={'display':'flex'}),])])


In [7]:

spx_site_success = spacex_df.groupby(['Launch Site'])['class'].sum().rename('Success Launches')
spx_site_success = pd.DataFrame(spx_site_success).reset_index()
spx_site_success.head()


,Launch Site,Success Launches
0,CCAFS LC-40,7
1,CCAFS SLC-40,3
2,KSC LC-39A,10
3,VAFB SLC-4E,4


In [8]:

spx_site_unsuccess = spacex_df[spacex_df['class']==0].groupby(['Launch Site'])
spx_site_unsuccess = spx_site_unsuccess['class'].count().rename('Unsuccess Launches').reset_index()
spx_site_unsuccess


,Launch Site,Unsuccess Launches
0,CCAFS LC-40,19
1,CCAFS SLC-40,4
2,KSC LC-39A,3
3,VAFB SLC-4E,6


In [9]:

# spx_site_success.append(spx_site_unsuccess['Unsuccess Launches'])
spx_site_launch_outcomes = pd.concat([spx_site_success, spx_site_unsuccess['Unsuccess Launches']], axis=1)
spx_site_launch_outcomes


,Launch Site,Success Launches,Unsuccess Launches
0,CCAFS LC-40,7,19
1,CCAFS SLC-40,3,4
2,KSC LC-39A,10,3
3,VAFB SLC-4E,4,6


TASK 4: Add a callback function to render the success-payload-scatter-chart scatter plot

In [10]:

def to_string(lista):
    for n in range(len(lista)):
        lista[n] = str(lista[n])
    return lista
    
to_string(pl)
# to_string(lbls)
print(pl, lbls)


['0', '960', '1920', '2880', '3840', '4800', '5760', '6720', '7680', '8640', '9600'] [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]


In [11]:

marks = dict(zip(lbls, pl))
marks


{0: '0',
 1: '960',
 2: '1920',
 3: '2880',
 4: '3840',
 5: '4800',
 6: '5760',
 7: '6720',
 8: '7680',
 9: '8640',
 10: '9600'}

In [12]:

# value = int(round(len(marks)/2))
value = marks[int(round(len(marks)/2))]
value


'5760'

In [13]:

spacex_df[spacex_df['Launch Site']=='CCAFS LC-40'].pivot_table(index='class', values='Launch Site', aggfunc = 'count').reset_index()


,class,Launch Site
0,0,19
1,1,7


In [14]:

rango = [525.0, 500.0, 677.0]
rango = pd.Series(rango)
spx_payload_success = spacex_df[(spacex_df['Payload Mass (kg)'] >= rango.min()) & (spacex_df['Payload Mass (kg)'] <= rango.max())][['Payload Mass (kg)', 'class']]
spx_payload_success.head()


,Payload Mass (kg),class
2,525.0,0
3,500.0,0
4,677.0,0
13,570.0,0
26,500.0,0


In [15]:

@app.callback(Output(component_id='success-site_pie-chart', component_property='figure'),
              Input(component_id='site-dropdown', component_property='value'))  

def charts(entered_site):
    
    if entered_site == 'ALL':
        fig_1 = px.pie(spx_site_success,'Launch Site', 'Success Launches',
                     title='Number of Success Launches per Site')
    else:
        spx_entered_site = spacex_df[spacex_df['Launch Site']==entered_site]
        spx_entered_site = spx_entered_site.pivot_table(index='class', values='Launch Site', 
                                                        aggfunc = 'count').reset_index()
        fig_1 = px.pie(spx_entered_site, 
                       'class', 
                       'Launch Site', 
                       title = 'Success (1) vs. Failed (0) Launches from ' + entered_site + ' site')
    return fig_1
        

@app.callback(Output('success-payload_scatter-chart', 'figure'),
              Input('payload-slider', 'value'))
              
def get_scatter_plot(entered_payload):
    
    Min = entered_payload[0]
    Max = entered_payload[1]
    spx_entered_payload = spacex_df[(spacex_df['Payload Mass (kg)'] >= Min) & (spacex_df['Payload Mass (kg)'] <= Max)]
    
    fig_2 = px.scatter(spx_entered_payload, 'Payload Mass (kg)', 'class', color = 'Launch Site')
    
    return fig_2

if __name__ == '__main__':
    app.run(port=8098, debug=True)
    # app.run(port=5051, debug=True)
    